# Fase 3 - Actividad 5: Prueba de Concepto (PoC) Lineal Unificada

Este notebook ejecuta una prueba rápida de una sola pasada (Single-pass execution) del pipeline completo de Machine Learning para los dos principales contendientes: **Regresión Logística y SVM Lineal (LinearSVC)**.

El objetivo principal **no es buscar el mejor modelo ni la mejor métrica**, sino **certificar la compilación de la arquitectura para ambos modelos**:
1. Que los nodos de transformación funcionen en cadena sin errores de dimensionalidad.
2. Que el flujo de texto crudo $\rightarrow$ submuestreo $\rightarrow$ vectorización $\rightarrow$ entrenamiento $\rightarrow$ predicción compile adecuadamente tanto para Logística como para SVM.
3. Que podamos obtener métricas correctamente.

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

sns.set_theme(style="whitegrid")

## 1. Carga de una Fracción Mínima de Datos

Para que esta Prueba de Concepto sea rápida, tomaremos solo una pequeña muestra del conjunto de entrenamiento/validación (por ejemplo, 1000 registros). Así validamos la arquitectura sin invertir horas de computación.

In [7]:
input_path = 'data/train_val_set.csv'
# Cargamos solo 1000 filas aleatorias para hacer la prueba ultra-rápida
df = pd.read_csv(input_path, sep=';').sample(n=1000, random_state=42)
df['texto_lineal'] = df['texto_lineal'].fillna('')

print(f"Registros cargados para la PoC: {df.shape[0]}")

X = df[['texto_lineal']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dimensiones Train: {X_train.shape}")
print(f"Dimensiones Test: {X_test.shape}")

Registros cargados para la PoC: 1000
Dimensiones Train: (800, 1)
Dimensiones Test: (200, 1)


## 2. Declaración de Nodos Compartidos

Definimos los pasos de preprocesamiento (Submuestreo y Vectorización) que serán idénticos para ambos modelos.

In [8]:
undersampler = RandomUnderSampler(random_state=42)
preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(max_features=1000), 'texto_lineal')
    ]
)

## 3. Prueba de Compilación: Regresión Logística

Ensamblamos el pipeline terminando con `LogisticRegression` y lo ejecutamos.

In [9]:
pipeline_log = Pipeline(steps=[
    ('undersampler', undersampler),
    ('vectorizer', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline_log.fit(X_train, y_train)
print("Pipeline LOGÍSTICO entrenado exitosamente.")

y_pred_log = pipeline_log.predict(X_test)
print("\n--- Reporte de Clasificación (Regresión Logística) ---")
print(classification_report(y_test, y_pred_log))

Pipeline LOGÍSTICO entrenado exitosamente.

--- Reporte de Clasificación (Regresión Logística) ---
              precision    recall  f1-score   support

           0       0.73      0.70      0.72        83
           1       0.79      0.82      0.81       117

    accuracy                           0.77       200
   macro avg       0.76      0.76      0.76       200
weighted avg       0.77      0.77      0.77       200



## 4. Prueba de Compilación: SVM Lineal (LinearSVC)

Repetimos el ensamblaje terminando con `LinearSVC` para garantizar que la API del modelo de Soporte Vectorial acepte sin errores nuestra matriz transformada.

In [10]:
pipeline_svm = Pipeline(steps=[
    ('undersampler', undersampler),
    ('vectorizer', preprocessor),
    ('classifier', LinearSVC(random_state=42, max_iter=2000))
])

pipeline_svm.fit(X_train, y_train)
print("Pipeline SVM entrenado exitosamente.")

y_pred_svm = pipeline_svm.predict(X_test)
print("\n--- Reporte de Clasificación (SVM Lineal) ---")
print(classification_report(y_test, y_pred_svm))

Pipeline SVM entrenado exitosamente.

--- Reporte de Clasificación (SVM Lineal) ---
              precision    recall  f1-score   support

           0       0.69      0.73      0.71        83
           1       0.80      0.76      0.78       117

    accuracy                           0.75       200
   macro avg       0.74      0.75      0.75       200
weighted avg       0.75      0.75      0.75       200

